# CMAPSS — Détection d'anomalies (RUL ≤ 30) — Comparaison 6 modèles

| Champ | Valeur |
|-------|--------|
| **Dataset** | CMAPSS NASA C-MAPSS |
| **Seuil anomalie** | RUL ≤ 30 cycles → état critique (faulty) |
| **Domaines** | FD001 → FD002 → FD003 → FD004 (4 domaines) |
| **Modèles** | EWC · HDC · TinyOL · KMeans · Mahalanobis · DBSCAN |
| **Sprint** | S22 — ⚠ Tous les résultats sont mockés |

> **Note** : Le seuil RUL = 30 cycles est le critère opérationnel standard C-MAPSS. Une unité avec RUL ≤ 30 est considérée en état d'anomalie nécessitant une intervention de maintenance immédiate. L'impact du seuil sur la détection est analysé en Section 6.


In [ ]:
# Section 1 — Setup + imports + constantes + mock data
import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# --- CWD navigation ---
_cwd = Path(".").resolve()
if _cwd.name == "cmapss_anomaly_detection":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import save_figure

FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/cmapss_anomaly_detection"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TASK_NAMES       = ["FD001", "FD002", "FD003", "FD004"]
MODEL_ORDER      = ["EWC", "HDC", "TinyOL", "KMeans", "Mahalanobis", "DBSCAN"]
RAM_BUDGET       = 65_536
FAULTY_THRESHOLD = 30

SCATTER_MARKERS: dict[str, tuple[str, str]] = {
    "EWC":         ("o", "#1f77b4"),
    "HDC":         ("s", "#ff7f0e"),
    "TinyOL":      ("^", "#2ca02c"),
    "KMeans":      ("D", "#d62728"),
    "Mahalanobis": ("P", "#9467bd"),
    "DBSCAN":      ("*", "#8c564b"),
}

# Mock data — toutes les expériences anomaly CMAPSS sont mockées
MOCK_DATA: dict[str, dict] = {
    "EWC":         {"auroc": 0.81, "f1": 0.72, "precision": 0.78, "recall": 0.67,
                   "ram_peak_bytes": 1171,  "inference_latency_ms": 0.020},
    "HDC":         {"auroc": 0.76, "f1": 0.68, "precision": 0.71, "recall": 0.64,
                   "ram_peak_bytes": 7700,  "inference_latency_ms": 0.050},
    "TinyOL":      {"auroc": 0.79, "f1": 0.70, "precision": 0.74, "recall": 0.66,
                   "ram_peak_bytes": 940,   "inference_latency_ms": 0.005},
    "KMeans":      {"auroc": 0.64, "f1": 0.52, "precision": 0.58, "recall": 0.47,
                   "ram_peak_bytes": 2200,  "inference_latency_ms": 0.120},
    "Mahalanobis": {"auroc": 0.62, "f1": 0.49, "precision": 0.55, "recall": 0.44,
                   "ram_peak_bytes": 1600,  "inference_latency_ms": 0.004},
    "DBSCAN":      {"auroc": 0.59, "f1": 0.44, "precision": 0.50, "recall": 0.39,
                   "ram_peak_bytes": 55000, "inference_latency_ms": 0.250},
}

results = MOCK_DATA.copy()

display(Markdown("### ⚠ Toutes les expériences anomaly CMAPSS sont mockées — valeurs fictives calibrées."))
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"FIGURES_DIR : {FIGURES_DIR}")
print(f"Seuil anomalie : RUL ≤ {FAULTY_THRESHOLD} cycles")

## Section 2 — Tableau AUROC / F1 / Précision / Rappel

In [ ]:
# Section 2 — Tableau comparatif anomaly detection
rows = []
SUPERVISED_MODELS   = ["EWC", "HDC", "TinyOL"]
UNSUPERVISED_MODELS = ["KMeans", "Mahalanobis", "DBSCAN"]

for model in MODEL_ORDER:
    r = results[model]
    fam = "Supervisé" if model in SUPERVISED_MODELS else "Non-supervisé"
    in_budget = "✓" if r["ram_peak_bytes"] <= RAM_BUDGET else "✗"
    rows.append({
        "Modèle":    model,
        "Famille":   fam,
        "AUROC ↑":   f"{r['auroc']:.3f}",
        "F1 ↑":      f"{r['f1']:.3f}",
        "Précision ↑": f"{r['precision']:.3f}",
        "Rappel ↑":  f"{r['recall']:.3f}",
        "RAM (Ko) ↓": f"{r['ram_peak_bytes']/1024:.1f}",
        "STM32 ≤64Ko": in_budget,
        "Lat (ms) ↓": f"{r['inference_latency_ms']:.3f}",
    })

df = pd.DataFrame(rows).set_index("Modèle")
display(Markdown(
    f"### Détection anomalies CMAPSS — RUL ≤ {FAULTY_THRESHOLD} cycles (6 modèles)\n"
    "*⚠ Toutes les valeurs sont mockées.*"
))
display(df)

best_auroc = max(MODEL_ORDER, key=lambda m: results[m]["auroc"])
print(f"\nMeilleur AUROC : {best_auroc} = {results[best_auroc]['auroc']:.3f}")

## Section 3 — Barplot AUROC + F1 (supervisé vs non-supervisé)

In [ ]:
# Section 3 — Barplot groupé AUROC + F1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

COLORS = {
    "EWC":         "#1f77b4",
    "HDC":         "#4a90d9",
    "TinyOL":      "#7eb8e8",
    "KMeans":      "#2ca02c",
    "Mahalanobis": "#5cc05c",
    "DBSCAN":      "#98df8a",
}

for ax_idx, (metric_key, metric_label) in enumerate([("auroc", "AUROC"), ("f1", "F1")]):
    ax   = axes[ax_idx]
    x    = np.arange(len(MODEL_ORDER))
    vals = [results[m][metric_key] for m in MODEL_ORDER]
    cols = [COLORS[m] for m in MODEL_ORDER]

    bars = ax.bar(x, vals, color=cols, width=0.6, edgecolor="white", linewidth=0.8)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.5, label="Baseline aléatoire (0.5)")

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.008,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    # Séparation supervisés / non-supervisés
    ax.axvline(2.5, color="gray", linestyle=":", linewidth=1.0, alpha=0.6)
    ax.text(1.0, 0.38, "Supervisé",     ha="center", fontsize=9, color="#1f77b4", style="italic")
    ax.text(4.0, 0.38, "Non-supervisé", ha="center", fontsize=9, color="#2ca02c", style="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_ORDER, fontsize=11)
    ax.set_ylabel(f"{metric_label} (test set)", fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.set_title(f"{metric_label} — Anomaly detection CMAPSS (RUL ≤ {FAULTY_THRESHOLD})",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)

fig.suptitle("⚠ Toutes les valeurs sont mockées", fontsize=9, color="gray", y=0.01)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "anomaly_auroc_f1_bar.png")
display(Image(str(FIGURES_DIR / "anomaly_auroc_f1_bar.png")))

## Section 4 — Courbes ROC (mock)

In [ ]:
# Section 4 — Courbes ROC mock (6 modèles)
np.random.seed(42)

def _mock_roc_curve(auroc_target: float, n_points: int = 100) -> tuple[np.ndarray, np.ndarray]:
    """Génère une courbe ROC fictive atteignant approximativement le AUROC cible."""
    fpr = np.linspace(0, 1, n_points)
    # Modèle : TPR = FPR^exp avec exp < 1 pour AUROC > 0.5
    # AUROC(exp) ≈ 1 / (1 + exp) → exp = 1/AUROC - 1
    e = max(0.05, 1.0 / max(auroc_target, 0.5) - 1)
    tpr = np.power(fpr, e)
    # Ajout de légère variabilité
    noise = np.random.normal(0, 0.015, size=n_points)
    tpr = np.clip(tpr + noise, 0, 1)
    tpr[0], tpr[-1] = 0.0, 1.0
    return fpr, tpr

fig, ax = plt.subplots(figsize=(9, 7))

ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Baseline aléatoire (AUROC=0.50)")

for model in MODEL_ORDER:
    target_auroc = results[model]["auroc"]
    fpr, tpr = _mock_roc_curve(target_auroc)
    marker, color = SCATTER_MARKERS[model]
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{model} (AUROC≈{target_auroc:.2f})")

ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title(
    f"Courbes ROC — CMAPSS Anomaly Detection (RUL ≤ {FAULTY_THRESHOLD} cycles)\n"
    "⚠ Courbes simulées — lancer expériences anomaly pour résultats réels",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "roc_curves_mock.png")
display(Image(str(FIGURES_DIR / "roc_curves_mock.png")))

## Section 5 — Scatter AUROC vs RAM (Gap 2, STM32 budget)

In [ ]:
# Section 5 — Scatter AUROC vs RAM (Gap 2)
fig, ax = plt.subplots(figsize=(8, 5))

ax.axvspan(0, RAM_BUDGET / 1024, alpha=0.07, color="green", label="Zone STM32 ≤ 64 Ko")
ax.axvline(RAM_BUDGET / 1024, color="red", linestyle="--", linewidth=1.5, label="Budget 64 Ko")

for name in MODEL_ORDER:
    r      = results[name]
    ram_kb = r["ram_peak_bytes"] / 1024
    auroc  = r["auroc"]
    marker, color = SCATTER_MARKERS[name]
    ax.scatter(ram_kb, auroc, marker=marker, color=color, s=130, zorder=5, label=name,
               edgecolor="black", linewidth=0.5)
    ax.annotate(name, xy=(ram_kb, auroc), xytext=(ram_kb * 1.04, auroc + 0.005), fontsize=9)

ax.set_xlabel("RAM peak (Ko)", fontsize=11)
ax.set_ylabel("AUROC ↑", fontsize=11)
ax.set_title(
    "Trade-off embarqué : RAM vs. AUROC\n"
    f"(CMAPSS anomaly RUL ≤ {FAULTY_THRESHOLD} — Gap 2 STM32 ≤ 64 Ko)",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=9, loc="lower right")
ax.set_ylim(0.4, 0.9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "scatter_auroc_vs_ram.png")
display(Image(str(FIGURES_DIR / "scatter_auroc_vs_ram.png")))

## Section 6 — Sensibilité au seuil RUL [UNIQUE CMAPSS]

Impact du seuil RUL sur l'AUROC de détection d'anomalies (EWC).

In [ ]:
# Section 6 — Sensibilité au seuil RUL (mock EWC)
# AUROC varie selon le seuil de classification (RUL ≤ threshold → anomalie)
THRESHOLDS   = [10, 15, 20, 25, 30, 40, 50, 60, 75]
# AUROC mock : pic autour de 30 cycles, optimum opérationnel standard
AUROC_BY_THRESHOLD = {
    "EWC":         [0.68, 0.73, 0.77, 0.80, 0.81, 0.79, 0.76, 0.73, 0.69],
    "Mahalanobis": [0.52, 0.55, 0.58, 0.61, 0.62, 0.61, 0.59, 0.57, 0.54],
    "KMeans":      [0.55, 0.58, 0.61, 0.63, 0.64, 0.63, 0.61, 0.58, 0.55],
}

fig, ax = plt.subplots(figsize=(10, 5))

MODELS_TO_SHOW = ["EWC", "Mahalanobis", "KMeans"]
COLORS_S = {"EWC": "#1f77b4", "Mahalanobis": "#9467bd", "KMeans": "#d62728"}
MARKERS_S = {"EWC": "o", "Mahalanobis": "P", "KMeans": "D"}

for model in MODELS_TO_SHOW:
    auroc_vals = AUROC_BY_THRESHOLD[model]
    ax.plot(THRESHOLDS, auroc_vals, marker=MARKERS_S[model], color=COLORS_S[model],
            linewidth=2, markersize=8, label=model)

ax.axvline(30, color="red", linestyle="--", linewidth=1.5, label="Seuil standard (30 cycles)")
ax.axvline(20, color="orange", linestyle=":",  linewidth=1.2, label="Seuil préventif (20 cycles)")

# Annotation pic EWC
ewc_max_idx = np.argmax(AUROC_BY_THRESHOLD["EWC"])
ax.annotate(
    f"EWC optimum\nRUL={THRESHOLDS[ewc_max_idx]} cycles\nAUROC={AUROC_BY_THRESHOLD['EWC'][ewc_max_idx]:.2f}",
    xy=(THRESHOLDS[ewc_max_idx], AUROC_BY_THRESHOLD["EWC"][ewc_max_idx]),
    xytext=(THRESHOLDS[ewc_max_idx] + 5, AUROC_BY_THRESHOLD["EWC"][ewc_max_idx] - 0.03),
    fontsize=9, color="#1f77b4",
    arrowprops={"arrowstyle": "->", "color": "#1f77b4"},
)

ax.set_xlabel("Seuil RUL (cycles) — anomalie si RUL ≤ seuil", fontsize=11)
ax.set_ylabel("AUROC ↑", fontsize=11)
ax.set_title(
    "Sensibilité au seuil RUL — AUROC vs threshold\n"
    "(CMAPSS anomaly — ⚠ valeurs mockées)",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=10)
ax.set_ylim(0.4, 0.9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "auroc_sensitivity_rul_threshold.png")
display(Image(str(FIGURES_DIR / "auroc_sensitivity_rul_threshold.png")))

print("Interprétation :")
print(f"  - Seuil trop bas (10 cycles) : peu de données 'anomalie' → AUROC faible")
print(f"  - Seuil trop haut (75 cycles): classes déséquilibrées → signal dilué")
print(f"  - Optimum opérationnel : 25-35 cycles (standard industriel C-MAPSS)")

## Section 7 — Comparaison CMAPSS vs PRONOSTIA (anomaly detection)

In [ ]:
# Section 7 — Tableau CMAPSS vs PRONOSTIA anomaly detection
# PRONOSTIA : seuil condition = dégradation bearing (classe fault vs normal)
PRONOSTIA_MOCK = {
    "EWC":         {"auroc": 0.84, "f1": 0.76, "ram_peak_bytes": 1171},
    "HDC":         {"auroc": 0.73, "f1": 0.65, "ram_peak_bytes": 7700},
    "TinyOL":      {"auroc": 0.81, "f1": 0.73, "ram_peak_bytes": 940},
    "KMeans":      {"auroc": 0.68, "f1": 0.58, "ram_peak_bytes": 2200},
    "Mahalanobis": {"auroc": 0.71, "f1": 0.61, "ram_peak_bytes": 1600},
    "DBSCAN":      {"auroc": 0.66, "f1": 0.54, "ram_peak_bytes": 55000},
}

comp_rows = []
for model in MODEL_ORDER:
    cr = results[model]
    pr = PRONOSTIA_MOCK[model]
    fam = "Supervisé" if model in ["EWC", "HDC", "TinyOL"] else "Non-supervisé"
    comp_rows.append({
        "Modèle":          model,
        "Famille":         fam,
        "CMAPSS AUROC":    f"{cr['auroc']:.3f}",
        "PRON AUROC":      f"{pr['auroc']:.3f}",
        "CMAPSS F1":       f"{cr['f1']:.3f}",
        "PRON F1":         f"{pr['f1']:.3f}",
        "Δ AUROC":         f"{pr['auroc'] - cr['auroc']:+.3f}",
    })

df_comp = pd.DataFrame(comp_rows).set_index("Modèle")
display(Markdown(
    "### CMAPSS vs PRONOSTIA — Anomaly Detection (6 modèles)\n"
    "*⚠ Toutes les valeurs sont mockées.*"
))
display(df_comp)

print("\nInterprétation :")
print("  PRONOSTIA AUROC généralement supérieur — dégradation bearing plus abrupte (signal plus clair).")
print("  CMAPSS : dégradation progressive turbofan → frontière anomalie/normal plus floue.")

## Conclusion

**Meilleur modèle pour détection d'anomalies CMAPSS** : EWC (AUROC≈0.81, F1≈0.72, RAM=1171 B)

| Observation | Détail |
|-------------|--------|
| Supervisés supérieurs | EWC > TinyOL > HDC >> non-supervisés |
| Optimum seuil RUL | 25-35 cycles (standard C-MAPSS opérationnel) |
| PRONOSTIA plus facile | AUROC PRON > AUROC CMAPSS pour tous les modèles |
| Seul EWC in-budget | RAM < 64 Ko, latence < 1 ms |

> **Note** : Ces résultats sont entièrement mockés. Lancer les scripts `anomaly_detection_config.yaml` pour obtenir les résultats réels. Voir `configs/cmapss_anomaly_detection_config.yaml`.